In [2]:
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pytz
import seaborn as sns

from usagestats import visual
from usagestats.bookingdb import BookingDB

folder = Path('/run/user/1000/gvfs/smb-share:server=istore,share=microscopy_group_share,user=jeromeb/Admin/Usage Stats/2026')
db_file = folder / 'bookings2.db'
local_tz = pytz.timezone("Europe/London")
start_date = datetime.fromisoformat('2025-11-04').replace(tzinfo=local_tz)
num_weeks = 52
end_date = start_date + timedelta(weeks=num_weeks)  # iso-format data yyyy-mm-dd
drop_bookings_type = ["service", "maintenance", "training"]
drop_instrument = ["Analysis PC1", "Analysis PC2", "Analysis PC3", "Analysis PC4"]

booking_types = ['default','service', 'maintenance', 'training']
divisions = ['Unknown','Cell Biology','Neurobiology','PNAC','Structural Studies','University']
groups = pd.read_csv(folder/'groups.csv')
users = pd.read_csv(folder/'users.csv')
training = pd.read_csv(folder/'training.csv')
calendars = pd.read_csv(folder/'calendars.csv')

# get a cookie from the curl command copied from firefox dev tools and save it in a cookie.txt file
with open('cookie.txt','r') as f:
    curl_str = f.read()

print(f"there are {len(calendars)} instruments")

# check that the calendars url are uniques
assert calendars['url'].is_unique, "Calendar URLs are not unique!"

training = training[~training["Email"].isna()]
training = training[~training["Group"].isna()]
training["Year"] = training["Date Requested"].apply(lambda x:str(x).split("/")[-1])
training["email"] = training["Email"].apply(lambda x:x.split("@")[0])
training["group"] = training["Group"].apply(lambda x:x.split(" ")[-1])
training["user"] = training["Name"]
training[["user","group","email"]]

users = pd.concat((training[["user","group","email"]],users[["user","group","email"]])).drop_duplicates()

# Load all calendars in the database
Path(db_file).unlink(True)
with BookingDB(db_file, curl_str=curl_str) as db:
    db.create_calendars_from_df(calendars)
    db.create_instruments()
    db.create_booking_types(booking_types)
    db.create_division(divisions)
    db.create_groups(groups)
    df, duplicates = db.create_events(users)


there are 31 instruments
load cal
Loading cached file Analysis PC1.ics
Loading cached file Analysis PC2.ics
Loading cached file Analysis PC3.ics
Loading cached file Analysis PC4.ics
Loading cached file Andor BC43.ics
Loading cached file Leica SP8.ics
Loading cached file Leica STED.ics
Loading cached file Nanoimager.ics
Loading cached file Nikon HCA.ics
Loading cached file Nikon ISIM.ics
Loading cached file Nikon NSTORM.ics
Loading cached file Nikon TI2.ics
Loading cached file Nikon TIRF.ics
Loading cached file Nikon W1.ics
Loading cached file Olympus VS200.ics
Loading cached file Olympus VS200.ics
Loading cached file OPM.ics
Loading cached file Spinning Disk.ics
Loading cached file Zeiss 2p.ics
Loading cached file Zeiss 710.ics
Loading cached file Zeiss 710 upright.ics
Loading cached file Zeiss 780.ics
Loading cached file Zeiss 780 UV.ics
Loading cached file Zeiss 780 UV.ics
Loading cached file Zeiss 880 Airyscan.ics
Loading cached file Zeiss 900 Airyscan AI.ics
Loading cached file Zei

In [4]:
users

,user,group,email
676,Yuening Li,Arseni,ynli
687,Esther Jones,Levin-Konigsberg,ejonesevans
688,Mark Walker,Ryskeldi-Falcon,mwalker
690,Andreea-Maria Aldea,Kornfeld,amaldea
691,Jake Cross,Levin-Konigsberg,jcross
...,...,...,...
188,Esther Jones Evans,Levin-Konigsberg,ejonesevans
189,Ryan Judy,Hedge,rjudy
190,Héctor Pizarro,Unknown,hpizarro
191,Victoria Hale,Allegretti,vhale


In [3]:
with BookingDB(db_file) as db:
    bookings = db.get_events_in_range(start_date, end_date)

newusers = bookings[bookings['group']=='Unknown'][['email', 'user','group','division']].drop_duplicates().reset_index()
print(f"Number of new users: {len(newusers)}")

# save the list to a csv file
new_users_path = folder/'new users.csv'

if not new_users_path.exists():
    newusers.to_csv(folder/'new users.csv')
else:
    print(f"File '{new_users_path.name}' exists.")
newusers

Number of new users: 200
File 'new users.csv' exists.


,index,email,user,group,division
0,0,ewatson@mrc-lmb.cam.ac.uk,Emily Watson,Unknown,Unknown
1,2,rwademan@mrc-lmb.cam.ac.uk,Richard Wademan,Unknown,Unknown
2,4,csia@mrc-lmb.cam.ac.uk,Chee Sia,Unknown,Unknown
3,6,mko@mrc-lmb.cam.ac.uk,Michelle Ko,Unknown,Unknown
4,7,afellows@mrc-lmb.cam.ac.uk,Alex Fellows,Unknown,Unknown
...,...,...,...,...,...
195,7645,fhawkings@mrc-lmb.cam.ac.uk,Frederick Russell Hawkings,Unknown,Unknown
196,7653,guilbaud@mrc-lmb.cam.ac.uk,Guillaume Guilbaud,Unknown,Unknown
197,8051,nmorrissey@mrc-lmb.cam.ac.uk,Nicole Morrissey,Unknown,Unknown
198,8201,rrk@mrc-lmb.cam.ac.uk,Robert Kay,Unknown,Unknown
